# LoRA: Low-Rank Adaptation - 실습 코드 2: LoRA 모듈을 직접 구현하기 (PyTorch)

- Tutorial ID: `expand-lora`
- Tutorial: LoRA: Low-Rank Adaptation
- Section ID: `expand-lora-code-2`
- Section: 실습 코드 2: LoRA 모듈을 직접 구현하기 (PyTorch)

---

> **이 노트북에 대해**
>
> 원래 실습 코드를 처음 배우는 사람도 한 줄씩 따라올 수 있도록 다시 정리했습니다.
> - 새로운 개념은 코드에 등장하기 **직전에** 먼저 말로 풀어서 설명합니다.
> - 가능한 곳마다 아주 작은 숫자로 된 예제를 직접 실행해서, "왜 이렇게 되는지"를 눈으로 확인합니다.
> - `torch`가 필요한 코드이므로 Colab이나 로컬/서버 Jupyter에서 실행해 주세요.


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: LoRA 모듈을 직접 구현하기 (PyTorch)
#
# 이 노트북은 "정답 코드를 한 번 실행해보고 끝내는" 용도가 아니라,
# LoRA 논문의 수식(W = W0 + B·A)이 실제 PyTorch 텐서 연산으로
# 어떻게 바뀌는지 한 줄씩 눈으로 확인하기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) 사전학습된 가중치 W0는 그대로 두고, 저랭크 행렬 A, B의 곱으로
#      업데이트(ΔW = B·A)를 표현하면 왜 학습 파라미터 수가 크게 줄어드는지 확인
#   2) nn.Linear를 감싸는 LoRALinear 모듈을 직접 만들어서
#      forward 계산 과정의 텐서 shape을 하나하나 추적
#   3) 모델 전체에서 원하는 레이어(q_proj, k_proj 등)만 골라
#      LoRA로 교체하고, "학습 가능한 파라미터"만 골라내는 방법 확인
#   4) 학습이 끝난 뒤 LoRA 가중치를 원래 가중치에 합치는(merge) 이유 이해
#
# 읽는 순서:
#   1) 아주 작은 행렬로 "저랭크 분해"가 무엇인지 먼저 감을 잡습니다. (Step 1)
#   2) nn.Linear의 가중치 shape 컨벤션을 복습합니다. (Step 2)
#   3) LoRALinear 클래스를 한 줄씩 뜯어보며 shape을 추적합니다. (Step 3~4)
#   4) 실제 모델(SmallLLM)에 LoRA를 적용해 파라미터가 얼마나 줄어드는지 봅니다. (Step 5~6)
#   5) LoRA 파라미터만 학습하는 루프를 만들고 미니 데모로 확인합니다. (Step 7~8)
#   6) 학습이 끝난 LoRA 가중치를 원본에 합치는 merge 과정을 확인합니다. (Step 9)
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "행렬의 shape이 어떻게 바뀌는지"와
#     "학습 가능한 파라미터 수가 왜 줄어드는지"에 집중해서 보세요.
#   - torch가 필요한 코드이므로 Colab이나 로컬/서버 Jupyter에서 실행하는 것을 권장합니다.
# ============================================================


## Step 0. 실습 환경 준비하기

이 노트북에서 사용하는 라이브러리는 사실상 **PyTorch(`torch`) 하나**뿐입니다.

| 이름 | 용도 |
|---|---|
| `torch` | 텐서 연산, 자동 미분(autograd)을 제공하는 핵심 라이브러리 |
| `torch.nn` | `nn.Linear`, `nn.Module`, `nn.Parameter` 등 신경망을 구성하는 부품들 |
| `math` | 파이썬 표준 라이브러리. 초기화 시 `√(2/d_in)` 같은 값을 계산할 때 사용 |

아래 셀에서 `torch.manual_seed(42)`로 랜덤 시드를 고정합니다.
난수(랜덤 값)를 쓰는 코드는 실행할 때마다 다른 숫자가 나올 수 있는데,
시드를 고정해두면 이 노트북을 몇 번 다시 실행해도 같은 숫자가 나와서 헷갈릴 일이 없습니다.


In [ ]:
import torch
import torch.nn as nn
import math

# 이 노트북에서 나오는 난수(랜덤 값)를 고정합니다.
# -> 다시 실행해도 항상 같은 숫자가 나오므로, 아래 설명과 실제 출력이 어긋나지 않습니다.
torch.manual_seed(42)

print("PyTorch 버전:", torch.__version__)


## Step 1. "저랭크(Low-Rank)"란 무엇인가? — 숫자로 먼저 감 잡기

LoRA(**Lo**w-**R**ank **A**daptation)라는 이름에 들어있는 **"저랭크(low-rank)"**부터 감을 잡아봅시다.

파인튜닝(fine-tuning)을 한다는 것은, 결국 사전학습된 가중치 행렬 `W0`를
`W0 + ΔW` 로 바꾸는 **변화량 ΔW**를 찾는 일입니다.

- **가장 단순한 방법**: `ΔW`를 `W0`와 똑같은 크기의 "꽉 찬(full-rank)" 행렬로 두고 통째로 학습
  → `W0`가 (1000, 1000) 크기라면, `ΔW`도 1,000,000개의 숫자를 새로 학습해야 합니다.
- **LoRA의 아이디어**: `ΔW`를 두 개의 "홀쭉한" 행렬의 곱으로 표현
  → `ΔW ≈ B @ A` (`B`는 (1000, r), `A`는 (r, 1000), `r`은 4~64 정도의 아주 작은 값)
  → 학습해야 할 숫자는 `1000*r + r*1000`개뿐!

이렇게 **"두 개의 작은 행렬의 곱"으로 큰 행렬을 근사하는 것**을 저랭크 분해(low-rank decomposition)라고 부릅니다.
먼저 큰 숫자로 얼마나 절감되는지 직접 계산해봅시다.


In [ ]:
# 예시: (1000, 1000) 크기의 가중치 변화량(ΔW)이 있다고 가정해봅시다.
demo_d_out, demo_d_in = 1000, 1000
demo_rank = 8   # r 값 (LoRA 논문/실무에서는 보통 4~64 사이를 씁니다)

full_params = demo_d_out * demo_d_in                            # ΔW를 통째로 저장/학습할 때 필요한 숫자 개수
lowrank_params = demo_d_in * demo_rank + demo_rank * demo_d_out  # 저랭크로 표현할 때 필요한 숫자 개수

print(f"(1) ΔW를 통째로 학습:            {full_params:,}개")
print(f"(2) 저랭크(rank={demo_rank})로 학습:      {lowrank_params:,}개")
print(f"    -> 원래 파라미터의 {lowrank_params/full_params*100:.2f}% 만으로 표현 가능!")


숫자만 봐서는 감이 잘 안 올 수 있으니, 실제로 두 개의 작은 행렬을 곱해서
정말로 "그보다 큰" 행렬 하나가 만들어지는지 직접 확인해봅시다.

행렬 곱셈은 `(m, r) @ (r, n) = (m, n)` 처럼 **가운데 숫자(r)가 서로 같아야 곱셈이 가능**하고,
결과는 **바깥쪽 숫자(m, n) 크기**의 행렬이 된다는 점을 기억해두세요. (뒤에서 계속 쓰입니다!)


In [ ]:
# 아주 작은 숫자로 "홀쭉한 행렬 두 개를 곱하면 큰 행렬이 나온다"는 것을 직접 확인해봅시다.
m, n, r = 6, 4, 3  # (6, 4) 크기의 행렬을 rank=3으로 근사한다고 가정

A_demo = torch.randn(m, r)   # (6, 3) : "홀쭉한" 행렬 1
B_demo = torch.randn(r, n)   # (3, 4) : "홀쭉한" 행렬 2

delta_W_demo = A_demo @ B_demo  # (6, 3) @ (3, 4) = (6, 4)

print("A_demo shape:", tuple(A_demo.shape))
print("B_demo shape:", tuple(B_demo.shape))
print("A_demo @ B_demo shape:", tuple(delta_W_demo.shape), " <- 정확히 (6, 4) 크기가 만들어졌습니다!")
print()
print("A_demo 파라미터 수:", A_demo.numel())
print("B_demo 파라미터 수:", B_demo.numel())
print(f"A+B 합계: {A_demo.numel() + B_demo.numel()}개  vs  원래 (6,4) 행렬을 그대로 저장: {m*n}개")


방금 예제에서는 `rank=3`이 `(6,4)`라는 작은 행렬 크기에 비해 그다지 작지 않다 보니,
저랭크로 표현한 파라미터 수(18+12=30개)가 오히려 원래 행렬(24개)보다 **더 많아졌습니다.**

이게 핵심입니다 — **저랭크 분해는 `rank`가 원래 행렬의 크기(d_in, d_out)에 비해
"충분히 작을 때"만 이득**입니다. 실제 LLM에서는

- 원래 가중치 행렬: 수백~수천 차원 (예: 4096 x 4096)
- LoRA의 rank: 4, 8, 16, 32처럼 아주 작은 값

이기 때문에, 맨 처음 (1000,1000) 예제처럼 압도적인 절감 효과가 나옵니다.

## Step 2. 시작 전에: `nn.Linear`의 가중치 shape 복습

`LoRALinear`를 구현하려면 먼저 PyTorch의 기본 `nn.Linear`가 내부적으로
가중치를 **어떤 shape으로** 들고 있는지 정확히 알아야 합니다. (여기서 많이들 헷갈립니다!)

`nn.Linear(in_features, out_features)`는 다음 수식으로 계산됩니다.

```
y = x @ W.T + b
```

여기서 가중치 `W`의 shape은 `(out_features, in_features)` 입니다.
**"입력, 출력" 순서가 아니라 "출력, 입력" 순서**라는 점에 주의하세요.
(그냥 `x @ W`가 아니라 `W`를 전치(`.T`)해서 곱하기 때문에 이런 shape을 가집니다.)

말로만 들으면 헷갈리니, 직접 만들어서 확인해봅시다.


In [ ]:
# in_features=4, out_features=6 인 Linear 레이어를 만들어봅니다.
linear_demo = nn.Linear(in_features=4, out_features=6)

print("linear_demo.weight의 shape:", tuple(linear_demo.weight.shape))
print("  -> (out_features, in_features) = (6, 4) 순서입니다.")
print("linear_demo.bias의 shape:", tuple(linear_demo.bias.shape))

x_demo = torch.randn(2, 4)  # batch_size=2, in_features=4 인 입력

# nn.Linear가 내부적으로 하는 계산을 "직접" 풀어서 재현해보면:
y_manual = x_demo @ linear_demo.weight.T + linear_demo.bias
y_builtin = linear_demo(x_demo)

print()
print("직접 계산한 결과와 nn.Linear(x)의 결과가 같은가?", torch.allclose(y_manual, y_builtin))


## Step 3. `LoRALinear` 모듈 설계하기

이제 준비가 끝났습니다. LoRA 논문의 핵심 수식을 다시 봅시다.

```
W = W0 + ΔW = W0 + B·A
```

이걸 우리가 구현할 `LoRALinear`에 맞게 조금 더 구체적으로 풀어보면 다음과 같습니다.

- **`W0`** : 기존 `nn.Linear`가 갖고 있던, 이미 학습된(사전학습된) 가중치. 학습 중에는 건드리지 않고 **얼립니다(freeze)**.
- **`A`** : shape `(d_in, rank)` — 입력을 `rank` 차원으로 "압축"하는 역할
- **`B`** : shape `(rank, d_out)` — 압축된 `rank` 차원을 다시 `d_out` 차원으로 "복원"하는 역할
- **`scale = alpha / rank`** : LoRA 업데이트의 크기를 조절하는 상수 (아래에서 자세히 설명)

즉 forward(순전파) 계산은 다음과 같이, **두 경로를 각각 계산해서 더하는 방식**입니다.
(`W0`에 `ΔW`를 미리 합쳐두지 않는다는 점이 중요합니다!)

```
y =  (원래 경로) x @ W0.T + b0
   + (LoRA 경로) (x @ A @ B) * scale
```

두 경로를 나누는 이유는, 학습 중에는 `A`, `B`만 업데이트하고 `W0`는 계속 그대로 유지해야 하기 때문입니다.
만약 처음부터 미리 합쳐버리면, 어디까지가 "원래 가중치"이고 어디부터가 "새로 배운 것"인지 구분할 수 없게 됩니다.

### 왜 `A`는 랜덤으로, `B`는 0으로 초기화할까?

- `B`를 전부 0으로 초기화하면, 학습을 시작하는 첫 순간에는 `A @ B = 0`이 되어 LoRA 경로의 출력이 0이 됩니다.
  → **학습 시작 시점에는 원래 모델과 완전히 동일하게 동작**합니다. (파인튜닝을 "아무것도 바뀌지 않은 안전한 상태"에서 시작할 수 있습니다.)
- 그런데 `A`까지 0으로 초기화하면 `A`의 그래디언트도 항상 0이 되어버려서 학습이 전혀 진행되지 않습니다.
  그래서 **`A`는 작은 랜덤값으로, `B`만 0으로** 초기화합니다.

### `scale = alpha / rank`는 왜 필요할까?

`rank`를 8에서 16으로 바꾸면, `A @ B`를 이루는 항의 개수가 늘어나면서 그 합(=결과값)이 평균적으로 커지는 경향이 있습니다.
그러면 `rank`를 바꿀 때마다 learning rate 같은 다른 하이퍼파라미터도 매번 다시 튜닝해야 하는 번거로움이 생깁니다.
`alpha`를 고정해두고 `scale = alpha / rank`로 나눠주면, `rank`를 바꾸더라도 업데이트의 "전체적인 크기"가
어느 정도 비슷하게 유지되어 다른 하이퍼파라미터를 크게 건드리지 않아도 됩니다.

이제 이 설명을 그대로 코드로 옮겨봅시다.


In [ ]:
class LoRALinear(nn.Module):
    """기존 nn.Linear 레이어를 감싸서 LoRA를 적용하는 모듈

    핵심 아이디어 (LoRA 논문 수식):
        W = W0 + ΔW = W0 + B·A
        - W0  : 사전학습된 가중치. 학습 중에는 얼려서(frozen) 그대로 사용
        - B·A : 학습으로 새로 배우는 "저랭크" 업데이트. 이 부분만 학습함

    forward 시에는 W를 직접 합치지 않고,
        y = x @ W0.T + (x @ A @ B) * scale
    처럼 "원래 경로"와 "LoRA 경로"를 각각 계산해서 더합니다.
    (학습이 끝난 뒤에만 merge_weights()로 두 경로를 하나로 합칩니다.)
    """

    def __init__(self, original_linear: nn.Linear, rank: int = 8,
                 alpha: float = 16.0, dropout: float = 0.0):
        super().__init__()

        # 1) 원래 있던 nn.Linear를 그대로 보관합니다. (바로 아래에서 freeze 처리)
        self.original = original_linear
        self.rank = rank
        self.alpha = alpha

        # 2) scale = alpha / rank
        #    rank를 바꿔도 LoRA 업데이트의 "전체적인 크기"가 비슷하게 유지되도록
        #    보정해주는 값입니다. (자세한 이유는 위 Step 3 설명 참고)
        self.scale = alpha / rank

        # nn.Linear.weight의 shape은 (out_features, in_features) 입니다.
        # 예) nn.Linear(4, 6).weight.shape == (6, 4)  (Step 2에서 확인한 내용)
        d_out, d_in = original_linear.weight.shape

        # 3) LoRA가 학습할 두 개의 저랭크 행렬 A, B를 만듭니다.
        #    - lora_A: (d_in, rank)  -> x(..., d_in)에 곱해서 (..., rank)로 "압축"
        #    - lora_B: (rank, d_out) -> (..., rank)에 곱해서 (..., d_out)로 "복원"
        #
        #    초기화가 중요합니다!
        #    - lora_A는 작은 랜덤값으로 초기화합니다. (완전히 0이면 그래디언트도 항상 0이라 학습이 안 됨)
        #    - lora_B는 전부 0으로 초기화합니다.
        #      -> 학습 시작 시점에는 A @ B = 0 이 되어 LoRA 경로의 출력이 0이 됩니다.
        #      -> 즉, 학습을 막 시작했을 때는 원래 모델과 완전히 동일하게 동작합니다.
        #      -> 학습이 진행되면서 서서히 LoRA가 모델의 동작을 바꿔나갑니다.
        self.lora_A = nn.Parameter(torch.randn(d_in, rank) * math.sqrt(2.0 / d_in))
        self.lora_B = nn.Parameter(torch.zeros(rank, d_out))

        # 4) LoRA 경로에 적용할 dropout (과적합 방지용, 선택사항)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # 5) 원래 가중치(W0)와 bias는 "얼립니다" (freeze).
        #    requires_grad=False로 설정하면 이 파라미터들은
        #    loss.backward()를 호출해도 그래디언트가 계산되지 않고,
        #    optimizer.step()을 호출해도 값이 바뀌지 않습니다.
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False

    def forward(self, x):
        # x의 shape: (..., d_in)  예) (batch_size, seq_len, d_in)

        # [경로 1] 원래 선형 변환 (W0는 frozen 상태이므로 그래디언트가 여기로는 흐르지 않음)
        original_out = self.original(x)   # (..., d_in) -> (..., d_out)

        # [경로 2] LoRA 경로: x -> (dropout) -> @A -> @B -> *scale
        #   x @ lora_A : (..., d_in)  @ (d_in, rank)  = (..., rank)
        #   ... @ lora_B: (..., rank) @ (rank, d_out) = (..., d_out)
        lora_out = self.dropout(x) @ self.lora_A @ self.lora_B * self.scale

        # 두 경로의 결과를 더해서 최종 출력을 만듭니다.
        # (학습 초기에는 lora_out이 0이므로 original_out과 완전히 같습니다.)
        return original_out + lora_out

    def merge_weights(self):
        """학습이 끝난 뒤, LoRA로 학습한 델타(ΔW)를 원래 가중치에 합쳐줍니다.

        왜 필요한가요?
        - 학습 중에는 원래 경로 + LoRA 경로를 "따로" 계산해야
          A, B만 학습하고 W0는 그대로 둘 수 있습니다.
        - 하지만 서비스에 배포할 때는 굳이 두 경로를 나눠서 계산할 필요가 없습니다.
          ΔW를 W0에 미리 더해두면, 추론(inference) 시에는
          원래의 nn.Linear 하나만 호출하면 되므로 계산이 더 빠르고 구조도 단순해집니다.
        """
        with torch.no_grad():
            # ΔW의 shape은 (d_out, d_in) 이어야 원래 weight와 더할 수 있습니다.
            # lora_A: (d_in, rank), lora_B: (rank, d_out) 이므로
            # (lora_B.T @ lora_A.T)의 shape은 (d_out, rank) @ (rank, d_in) = (d_out, d_in)
            delta_W = (self.lora_B.T @ self.lora_A.T) * self.scale
            self.original.weight.data += delta_W
        return self.original


## Step 4. 작은 예제로 `LoRALinear`가 제대로 동작하는지 확인하기

이론상 다음 두 가지가 반드시 성립해야 합니다.

1. 방금 만든 `LoRALinear`는 **학습 전(=`lora_B`가 0일 때)에는 원래 `nn.Linear`와 완전히 같은 출력**을 내야 합니다.
2. **원래 가중치(`W0`, `b0`)는 `requires_grad=False`로 얼려져 있어야** 하고, **`lora_A`, `lora_B`만 학습 가능**해야 합니다.

작은 숫자로 직접 확인해봅시다.


In [ ]:
torch.manual_seed(0)

# in_features=8, out_features=4 인 원래 레이어를 하나 만듭니다.
original_layer = nn.Linear(8, 4)

# rank=2로 LoRA를 적용해봅니다.
lora_layer = LoRALinear(original_layer, rank=2, alpha=4.0)

x_check = torch.randn(3, 8)  # batch_size=3, d_in=8

out_original = original_layer(x_check)
out_lora = lora_layer(x_check)

print("[확인 1] 학습 전(=lora_B가 0일 때) 출력이 서로 같은가?")
print("  원래 레이어 출력[0]:", out_original[0])
print("  LoRA 레이어 출력[0]:", out_lora[0])
print("  -> allclose:", torch.allclose(out_original, out_lora))

print()
print("[확인 2] 어떤 파라미터가 학습 가능(requires_grad=True)한가?")
for name, param in lora_layer.named_parameters():
    print(f"  {name:25s} shape={str(tuple(param.shape)):12s} requires_grad={param.requires_grad}")


`lora_B`를 실제로 0이 아닌 값으로 바꾸면 출력이 정말 달라지는지도 확인해봅시다.
(실제 학습에서는 `optimizer.step()`이 이 역할을 자동으로 해줍니다.)


In [ ]:
with torch.no_grad():
    lora_layer.lora_B.copy_(torch.randn_like(lora_layer.lora_B) * 0.1)

out_lora_after = lora_layer(x_check)

print("lora_B를 0이 아닌 값으로 바꾼 뒤 출력[0]:", out_lora_after[0])
print("원래 레이어 출력과 같은가?", torch.allclose(out_original, out_lora_after))
print("-> lora_B가 0이 아니게 되니, 이제 LoRA 경로가 실제로 출력에 영향을 주기 시작했습니다.")


## Step 5. 모델 안에서 원하는 레이어만 골라 LoRA로 교체하기

지금까지는 `nn.Linear` 하나에만 LoRA를 적용해봤습니다.
실제로는 Transformer 모델 안에 있는 수십~수백 개의 `nn.Linear` 중에서,
**"이름에 `q_proj`, `k_proj`, `v_proj`, `o_proj`가 들어간 레이어들만"** 골라서 LoRA로 바꿔치기해야 합니다.

`apply_lora_to_model` 함수가 하는 일은 크게 3단계입니다.

1. **모델 전체를 먼저 얼립니다 (freeze).** 모든 파라미터의 `requires_grad`를 `False`로 만듭니다.
   → 이 단계가 없으면, LoRA로 교체되지 않은 나머지 레이어들(예: FFN, 임베딩)이
     여전히 학습 가능한 상태로 남아버려서, "LoRA는 극히 일부 파라미터만 학습한다"는
     원래 목적이 깨지게 됩니다.
2. `model.named_modules()`로 모델 안의 모든 서브모듈을 `"layers.0.q_proj"`처럼
   점(`.`)으로 이어진 이름과 함께 하나씩 순회합니다.
3. 이름에 원하는 키워드가 포함되어 있고 `nn.Linear` 타입이면,
   `LoRALinear`로 감싼 뒤 `setattr`로 실제 모델 트리 안의 그 위치를 교체합니다.

`setattr(parent, "q_proj", lora_module)`처럼 **이름의 마지막 조각만** 바꿔치기하기 때문에,
`model.layers[0].ffn`처럼 감싸지 않은 나머지 구조는 그대로 유지됩니다.


In [ ]:
def apply_lora_to_model(model, rank=8, alpha=16.0, target_names=None):
    """모델에서 이름이 target_names에 해당하는 nn.Linear 레이어들을 찾아 LoRA로 교체합니다.

    핵심: LoRA를 적용하기 전에 먼저 모델 "전체"를 얼립니다.
         그래야 최종적으로 학습 가능한 파라미터가 lora_A, lora_B뿐이라는 것이 보장됩니다.
         (개별 LoRALinear가 자기 자신의 원래 가중치만 얼리는 것으로는 부족합니다.
          LoRA를 적용하지 않은 다른 레이어들도 얼려야 하기 때문입니다.)
    """
    if target_names is None:
        target_names = ["q_proj", "k_proj", "v_proj", "o_proj"]

    # 1) 먼저 모델 전체를 동결(freeze)합니다.
    for param in model.parameters():
        param.requires_grad = False

    lora_params = []
    total_params = 0

    # 2) named_modules()로 모델 안의 모든 서브모듈을 (이름, 모듈) 쌍으로 순회합니다.
    #    예) "layers.0.q_proj", "layers.0.ffn.0" 처럼 점(.)으로 계층을 표현한 이름이 나옵니다.
    for name, module in model.named_modules():
        # 이름에 target_names 중 하나라도 포함되어 있고, nn.Linear 타입인 경우만 교체
        if any(t in name for t in target_names) and isinstance(module, nn.Linear):
            # 2-1) 기존 nn.Linear를 LoRALinear로 감쌉니다.
            lora_module = LoRALinear(module, rank=rank, alpha=alpha)

            # 2-2) 실제로 모델 트리에서 이 레이어를 교체합니다.
            #      name이 "layers.0.q_proj"라면
            #        parts = ["layers", "0", "q_proj"]
            #      -> model.layers[0] 까지 내려간 뒤, 그 객체의 "q_proj" 속성을 교체합니다.
            parts = name.split('.')
            parent = model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            setattr(parent, parts[-1], lora_module)

            # 2-3) 이 레이어에서 새로 생긴 학습 가능한 파라미터(lora_A, lora_B)를 모아둡니다.
            #      -> 나중에 optimizer에는 "이 리스트만" 전달할 것입니다.
            lora_params.extend([lora_module.lora_A, lora_module.lora_B])
            n_lora = lora_module.lora_A.numel() + lora_module.lora_B.numel()
            total_params += n_lora
            print(f"  LoRA 적용: {name:20s} "
                  f"(원래 weight shape={tuple(module.weight.shape)}, rank={rank}, 추가 파라미터={n_lora:,}개)")

    print(f"\n총 LoRA 파라미터: {total_params:,}개")
    return model, lora_params


## Step 6. 미니 언어모델에 LoRA 적용해보기

실제 LLM은 너무 크니, 구조만 비슷하게 흉내 낸 아주 작은 모델 `SmallLLM`을 만들어서 실습합니다.

> **주의**: 이 모델의 self-attention은 진짜 Transformer의 attention을 아주 단순화한 버전입니다.
> (멀티헤드로 나누는 것, causal mask, 위치 인코딩(positional encoding) 등은 생략했습니다.)
> "LoRA를 q_proj/k_proj/v_proj/o_proj에 적용한다"는 것이 실제로 어떤 레이어를 가리키는지
> 보여주는 용도이지, 제대로 된 언어모델을 학습시키는 것이 목적이 아닙니다.

구조는 다음과 같습니다.

- `embed` : 토큰 ID → 벡터로 바꾸는 임베딩 레이어
- `layers` : `n_layers`개의 블록. 각 블록은
  - `q_proj`, `k_proj`, `v_proj` : Query/Key/Value를 만드는 선형변환 **(← LoRA 적용 대상!)**
  - 단순화된 self-attention: `softmax(Q·K^T / √d) · V`
  - `o_proj` : attention 결과를 다시 합쳐주는 선형변환 **(← LoRA 적용 대상!)**
  - `ffn` : Linear → GELU → Linear로 이루어진 피드포워드 네트워크 (← 기본 설정에서는 LoRA 적용 대상 아님)
- `lm_head` : 마지막 hidden state를 vocab 크기의 logits로 바꾸는 레이어 (← LoRA 적용 대상 아님)


In [ ]:
class SmallLLM(nn.Module):
    """실습용으로 아주 단순화한 미니 언어모델

    진짜 GPT/LLaMA류 모델과 레이어 이름(q_proj, k_proj, v_proj, o_proj, ffn)은 비슷하게 맞췄지만,
    - 멀티헤드로 나누지 않고
    - causal mask(미래 토큰을 못 보게 가리는 처리)도 없고
    - 위치 인코딩(positional encoding)도 없는
    아주 단순화된 버전입니다. "LoRA를 어떤 레이어에 적용하는지" 보여주는 용도로만 사용하세요.
    """

    def __init__(self, vocab_size=32000, d_model=512, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                "q_proj": nn.Linear(d_model, d_model),
                "k_proj": nn.Linear(d_model, d_model),
                "v_proj": nn.Linear(d_model, d_model),
                "o_proj": nn.Linear(d_model, d_model),
                "ffn": nn.Sequential(
                    nn.Linear(d_model, d_model * 4),
                    nn.GELU(),
                    nn.Linear(d_model * 4, d_model),
                ),
            }))

        # 마지막 hidden state를 vocab_size 크기의 logits로 바꿔주는 레이어
        # (다음 토큰이 어떤 단어일지 예측하려면 필요합니다)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: (batch_size, seq_len) 크기의 토큰 ID들
        x = self.embed(x)  # -> (batch_size, seq_len, d_model)

        for layer in self.layers:
            # ---- 아주 단순화된 self-attention ----
            q = layer["q_proj"](x)   # (batch, seq, d_model)
            k = layer["k_proj"](x)
            v = layer["v_proj"](x)

            # 토큰끼리 서로 얼마나 관련 있는지 점수를 계산 (Q와 K의 내적)
            d_model = x.size(-1)
            scores = q @ k.transpose(-2, -1) / math.sqrt(d_model)  # (batch, seq, seq)
            attn_weights = torch.softmax(scores, dim=-1)            # 점수를 0~1 확률로 변환
            attn_out = attn_weights @ v                              # (batch, seq, d_model)

            x = layer["o_proj"](attn_out) + x   # attention 결과 + 잔차 연결(residual connection)
            # ---- 피드포워드 네트워크 ----
            x = layer["ffn"](x) + x             # FFN 결과 + 잔차 연결(residual connection)

        logits = self.lm_head(x)   # (batch, seq, vocab_size)
        return logits


In [ ]:
model = SmallLLM(vocab_size=32000, d_model=512, n_layers=4)

total_before = sum(p.numel() for p in model.parameters())
print(f"LoRA 적용 전 전체 파라미터 수: {total_before:,}개")
print(f"(참고: 지금은 전부 requires_grad=True 라서, 이 상태로 학습하면 {total_before:,}개를 전부 업데이트해야 합니다)")

# 모델이 실제로 잘 동작하는지 짧게 확인 (토큰 ID 몇 개를 넣어봄)
test_ids = torch.randint(0, 32000, (2, 10))  # batch_size=2, seq_len=10
test_out = model(test_ids)
print()
print("forward 출력 shape:", tuple(test_out.shape), " (batch_size, seq_len, vocab_size) 이어야 정상")


In [ ]:
print("LoRA 적용 중...\n")
model, lora_params = apply_lora_to_model(model, rank=8, alpha=16.0)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print()
print(f"전체 파라미터 수:      {total_before:,}개")
print(f"학습 가능 파라미터 수:  {trainable:,}개")
print(f"학습 가능 비율:        {trainable/total_before*100:.3f}%")
print()
print("(위의 Step 5에서 '모델 전체를 먼저 얼린다'는 단계를 빼먹으면,")
print(" ffn/embed/lm_head까지 전부 학습 가능한 상태로 남아서 이 비율이 훨씬 커집니다.")
print(" 궁금하다면 apply_lora_to_model 안의 freeze 코드를 주석 처리하고 다시 실행해서 비교해보세요.)")


## Step 7. LoRA 파라미터만 학습하는 학습 루프

일반적인 PyTorch 학습 루프와 거의 똑같습니다. 다른 점은 딱 하나,
`optimizer`에 `model.parameters()`(모델의 모든 파라미터)가 아니라
`lora_params`(방금 모아둔 `lora_A`, `lora_B`들)만 넘겨준다는 점입니다.

optimizer가 모르는 파라미터는 `loss.backward()`로 그래디언트가 계산되더라도
`optimizer.step()`으로 업데이트되지 않습니다. (물론 얼려둔 파라미터는 애초에
`requires_grad=False`라서 그래디언트 자체가 계산되지 않습니다.)


In [ ]:
def train_with_lora(model, lora_params, train_loader, epochs=3, lr=1e-4):
    """LoRA 파라미터(lora_A, lora_B)만 학습하는 루프"""

    # [핵심] optimizer에 "lora_params"만 전달합니다.
    #   model.parameters() 전체가 아니라, apply_lora_to_model()이 모아준
    #   lora_A / lora_B 리스트만 넘기기 때문에, 나머지 파라미터는 계속 얼어붙어 있습니다.
    optimizer = torch.optim.AdamW(lora_params, lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids, labels = batch

            logits = model(input_ids)  # (batch, seq_len, vocab_size)

            # 다음 토큰 예측 문제로 보고 cross entropy loss를 계산합니다.
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),  # (batch*seq_len, vocab_size)
                labels.view(-1)                     # (batch*seq_len,)
            )

            optimizer.zero_grad()
            loss.backward()     # lora_A, lora_B에 대해서만 그래디언트가 계산됨
            optimizer.step()    # lora_A, lora_B만 실제로 업데이트됨

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}: loss={avg_loss:.4f}")
        scheduler.step()


## Step 8. 가짜 데이터로 학습이 실제로 되는지 확인하기

진짜 텍스트 데이터를 준비하는 대신, **랜덤한 토큰 ID**로 아주 작은 가짜 데이터셋을
만들어서 학습 루프가 실제로 잘 도는지 확인해봅시다.

데이터가 가짜(랜덤)라서 loss가 "의미 있게" 줄어들지는 않겠지만, 최소한 다음 세 가지는 확인할 수 있습니다.

1. 학습 루프가 에러 없이 끝까지 실행되는가
2. `lora_B`가 실제로 0에서 다른 값으로 바뀌는가 (= 제대로 학습되고 있는가)
3. `original.weight`는 학습 전후로 정말 그대로인가 (= 제대로 얼려져 있는가)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

vocab_size, seq_len, n_samples, batch_size = 32000, 16, 8, 4

# 랜덤 토큰 ID로 가짜 입력/정답 데이터를 만듭니다. (실제로는 텍스트를 토큰화한 결과가 들어갑니다)
fake_input_ids = torch.randint(0, vocab_size, (n_samples, seq_len))
fake_labels = torch.randint(0, vocab_size, (n_samples, seq_len))

train_dataset = TensorDataset(fake_input_ids, fake_labels)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 학습 전, 비교용으로 특정 LoRA 레이어의 lora_B와 원래 weight를 미리 복사해둡니다.
first_lora_layer = model.layers[0]["q_proj"]  # apply_lora_to_model이 LoRALinear로 교체해둔 레이어
lora_B_before = first_lora_layer.lora_B.clone()
original_weight_before = first_lora_layer.original.weight.clone()

print("학습 전 lora_B가 전부 0인가?", torch.all(lora_B_before == 0).item())


In [ ]:
train_with_lora(model, lora_params, train_loader, epochs=2, lr=1e-3)


In [ ]:
lora_B_after = first_lora_layer.lora_B
original_weight_after = first_lora_layer.original.weight

print("학습 후 lora_B가 여전히 전부 0인가?", torch.all(lora_B_after == 0).item(),
      " (False가 나와야 정상: 학습되면서 값이 바뀌었다는 뜻)")
print("original.weight가 학습 전후로 완전히 같은가?", torch.equal(original_weight_before, original_weight_after),
      " (True가 나와야 정상: 제대로 얼려서 그대로라는 뜻)")


## Step 9. 학습이 끝난 후: LoRA 가중치를 원래 가중치에 합치기 (merge)

학습 중에는 "원래 경로 + LoRA 경로"를 항상 따로 계산해야 `A`, `B`만 업데이트할 수 있습니다.
하지만 학습이 끝나고 나면, 굳이 두 경로를 나눠서 계산할 이유가 없습니다.

`merge_weights()`는 학습된 `ΔW = B.T @ A.T * scale`를 원래 `weight`에 더해서,
LoRA 이전과 완전히 동일한 구조(순수한 `nn.Linear` 하나)로 되돌려줍니다.

**장점**
- 추론(inference) 시 계산량이 줄어듭니다. (LoRA 경로의 추가 행렬곱이 사라짐)
- 배포할 때 `LoRALinear` 같은 커스텀 클래스 없이, 표준 `nn.Linear`로만 이루어진
  모델을 그대로 저장/배포할 수 있습니다.

merge 전후로 출력이 정말 똑같은지 작은 예제로 확인해봅시다.


In [ ]:
torch.manual_seed(1)

# 학습이 어느 정도 진행된 상태를 흉내 내기 위해, lora_B에 임의의 값을 넣어봅니다.
test_linear = nn.Linear(6, 4)
lora_test = LoRALinear(test_linear, rank=2, alpha=8.0)
with torch.no_grad():
    lora_test.lora_B.copy_(torch.randn_like(lora_test.lora_B) * 0.1)

x_test = torch.randn(3, 6)

out_before_merge = lora_test(x_test)          # "원래 경로 + LoRA 경로" 방식으로 계산
merged_linear = lora_test.merge_weights()      # ΔW를 weight에 합침 -> 순수 nn.Linear 반환
out_after_merge = merged_linear(x_test)        # 이제는 원래 nn.Linear 계산 한 번뿐

print("merge 전 출력[0]:", out_before_merge[0])
print("merge 후 출력[0]:", out_after_merge[0])
print("두 출력이 (거의) 같은가?", torch.allclose(out_before_merge, out_after_merge, atol=1e-6))
print()
print("merge 후 레이어 타입:", type(merged_linear).__name__, " <- 이제 순수한 nn.Linear입니다")


## 정리

| 개념 | 한 줄 요약 |
|---|---|
| `W = W0 + B·A` | 가중치 변화량을 두 개의 저랭크 행렬 곱으로 표현 |
| `rank` | `A`, `B`의 "허리 굵기". 작을수록 파라미터가 적어짐 (보통 4~64) |
| `alpha`, `scale = alpha/rank` | LoRA 업데이트의 크기를 조절하는 하이퍼파라미터 |
| `lora_A`=랜덤, `lora_B`=0 초기화 | 학습 시작 시점에 원래 모델과 동일하게 출발하기 위함 |
| `requires_grad=False` | 원래 가중치를 얼려서(freeze) 학습되지 않게 함 |
| `target_names` | 모델 전체가 아니라 원하는 레이어(q_proj 등)에만 LoRA 적용 |
| `optimizer(lora_params)` | 학습 가능한 파라미터를 lora_A/B로만 제한 |
| `merge_weights()` | 학습 후 ΔW를 원래 가중치에 합쳐 배포용으로 단순화 |

이 노트북 설정(SmallLLM, rank=8) 그대로 실행하면, 학습 가능한 파라미터 비율은
전체의 **약 0.3%** 수준까지 줄어듭니다. 파라미터 수가 억 단위인 실제 LLM에서는
이 비율이 1% 미만, 심지어 0.1% 미만까지도 내려갈 수 있습니다.

### 직접 실험해볼 것들
- `rank`를 2, 8, 32로 바꿔가며 `학습 가능 파라미터 수`가 어떻게 달라지는지 비교해보세요.
- `apply_lora_to_model`을 호출할 때 `target_names`에 `"ffn"`을 추가해서, FFN 레이어까지
  LoRA를 적용하면 파라미터 수가 얼마나 늘어나는지 확인해보세요.
- `alpha`만 바꾸고 `rank`는 고정했을 때, `scale` 값과 학습 초반 loss가 어떻게 달라지는지 관찰해보세요.
- Step 5의 "모델 전체를 먼저 얼리는" 코드를 주석 처리하고 다시 실행해서, 학습 가능 비율이
  왜 커지는지 직접 확인해보세요.
